# Gtransporte: fragilidad de la red troncal de TransMilenio

**Pregunta:** ¿qué tan frágil es la red troncal de TransMilenio y cómo se compara con la red vial?

La red se construye desde el GTFS de Transmilenio S.A. en dos espacios (reglas en `docs/specs/gtransporte.md`):

- **Espacio L:** estaciones unidas si son consecutivas sobre la vía. Es la infraestructura y se compara con Gvial.
- **Espacio P:** estaciones unidas si un bus va de una a la otra sin transbordo. Es la oferta de viajes directos.

Ambos se analizan con el mismo módulo que Gvial (`src/pipeline_redes.py`). Corre igual en Google Colab y en un equipo local.

## 1. Entorno y rutas

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

VERSIONES = {"osmnx": "2.1.1", "networkx": "3.7"}

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[f"{p}=={v}" for p, v in VERSIONES.items()]], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/proyecto-redes-bogota")
    RUTAS = {"src": BASE / "src", "raw": BASE / "data" / "raw" / "gtransporte",
             "salida": BASE / "resultados" / "gtransporte",
             "gvial": BASE / "resultados" / "gvial"}
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUTAS = {"src": RAIZ / "src", "raw": RAIZ / "data" / "raw" / "gtransporte",
             "salida": RAIZ / "data" / "salidas_locales" / "resultados" / "gtransporte",
             "gvial": RAIZ / "data" / "salidas_locales" / "resultados" / "gvial"}

RUTAS["salida"].mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(RUTAS["src"]))
for nombre, ruta in RUTAS.items():
    print(f"{nombre:8s} {ruta}")

In [ ]:
import logging

import networkx as nx
import pandas as pd

import gtfs_transmilenio as gt
import pipeline_redes as pr

logging.basicConfig(level=logging.INFO, format="%(message)s")
print("networkx", nx.__version__)

## 2. Parámetros

In [ ]:
FUENTE = {
    "url": "https://files.mobilitydatabase.org/mdb-3358/mdb-3358-202607130343/mdb-3358-202607130343.zip",
    "sha256": "bc1f00f2a4e36417655b95a4f4835e76457e7a26a140179f04f10adfc2a458e1",
    "archivo": "gtfs_transmilenio_2026-07-13.zip",
    "fecha": "2026-07-13",
}
PARAM = {
    "tolerancia_m": 80,          # distancia máxima de una estación al trazado
    "max_arista_m": 4000,        # aristas más largas son artefactos de bucles
    "franja_pico": (6, 9),       # hora pico AM para la frecuencia del espacio P
    "min_km_corredor": 2.0,
    "top_n": 10,
    "cortes_dep_vel": [0.6, 2],
    "etiquetas_dep_vel": ["subutilizados", "estructurales", "atraen por velocidad"],
    "umbral_fc": 0.5,
    "ataque": {"pasos": 40, "frac": 0.01, "reps_azar": 5},
    "robustez": {"pasos": 10, "frac": 0.01},
    "offsets_replicas": [0, 100, 200],
}

## 3. Construcción de los grafos

El GTFS se descarga de una dirección fija y se verifica su huella SHA-256: si el archivo no es exactamente la copia del 2026-07-13, el notebook se detiene.

In [ ]:
zip_gtfs = gt.obtener_gtfs(FUENTE["url"], RUTAS["raw"] / FUENTE["archivo"], FUENTE["sha256"])
feed = gt.leer_feed(zip_gtfs)
fuente = f"GTFS Transmilenio S.A., Mobility Database mdb-3358"
GL = gt.construir_L(feed, PARAM["tolerancia_m"], PARAM["max_arista_m"], fuente, FUENTE["fecha"])
GP = gt.construir_P(feed, GL, PARAM["franja_pico"], fuente, FUENTE["fecha"])
for G in (GL, GP):
    pr.validar_contrato(G)
    nx.write_graphml(G, RUTAS["salida"] / f"{G.graph['red'].replace(' ', '_')}.graphml")
    print(f"{G.graph['red']}: {G.number_of_nodes()} estaciones, {G.number_of_edges()} aristas")
print("Aristas L con tiempo por velocidad mediana:", GL.graph["aristas_sin_tiempo_programado"])
gt.resumen_troncales(GL)

## 4. Estructura, intermediación y concentración

In [ ]:
REDES = {"L": GL, "P": GP}
fuentes = {e: list(G.nodes) for e, G in REDES.items()}   # red pequeña: todas las estaciones como origen
est, eb, tramos, conc = {}, {}, {}, {}
for e, G in REDES.items():
    est[e] = pr.estructura(G, n_fuentes=G.number_of_nodes())
    eb[e] = {"eb": pr.intermediacion(G, "travel_time"), "eb_len": pr.intermediacion(G, "length")}
    tramos[e] = pr.tabla_tramos(G, eb[e])
    conc[e] = pr.concentracion(tramos[e]["eb"])
    tramos[e].to_csv(RUTAS["salida"] / f"tramos_{e}.csv", index=False)

tabla_est = pd.DataFrame({e: {**est[e], **conc[e], "spearman_pesos": pr.spearman(tramos[e]["eb"], tramos[e]["eb_len"])}
                          for e in REDES})
tabla_est.to_csv(RUTAS["salida"] / "estructura_concentracion.csv")
tabla_est.round(4)

## 5. Tramos y troncales críticos (espacio L)

Los tramos con mayor intermediación y la carga de cada troncal, con el índice de dependencia de velocidad (cortes 0,6 y 2, iguales a Gvial).

In [ ]:
nombre_est = nx.get_node_attributes(GL, "nombre")
top_tramos = (tramos["L"].nlargest(PARAM["top_n"], "eb")
              .assign(origen=lambda t: t["u"].map(nombre_est), destino=lambda t: t["v"].map(nombre_est)))
top_tramos[["origen", "destino", "nombre", "length_m", "eb"]]

In [ ]:
corr_L = pr.corredores(tramos["L"], {"carga_t": "eb", "carga_l": "eb_len"}, min_km=PARAM["min_km_corredor"])
dep_L = pr.dependencia_velocidad(corr_L, "carga_t", "carga_l", PARAM["cortes_dep_vel"], PARAM["etiquetas_dep_vel"])
dep_L.to_csv(RUTAS["salida"] / "troncales_L.csv")
dep_L.sort_values("carga_t", ascending=False).round(4)

## 6. Resiliencia

Mismas estrategias que Gvial: fallas al azar (5 réplicas), ranking fijo de intermediación y ataque adaptativo. Por el tamaño de la red, la intermediación del ataque adaptativo es exacta. La fracción crítica f_c es donde la componente gigante S cae a 0,5.

In [ ]:
A, u = PARAM["ataque"], PARAM["umbral_fc"]
curvas, fc, E0 = {}, {}, {}
for e, G in REDES.items():
    E0[e] = pr.eficiencia(G, fuentes[e])
    k = G.number_of_nodes()
    cs = [pr.ataque(G, "aleatorio", fuentes[e], E0[e], A["pasos"], A["frac"], seed=s)[0] for s in range(A["reps_azar"])]
    cs.append(pr.ataque(G, "eb_inicial", fuentes[e], E0[e], A["pasos"], A["frac"], eb_inicial=eb[e]["eb"])[0])
    cs.append(pr.ataque(G, "eb_recalculada", fuentes[e], E0[e], A["pasos"], A["frac"], k_recalculo=k)[0])
    curvas[e] = pd.concat(cs, ignore_index=True).assign(espacio=e)
    azar = curvas[e][curvas[e]["estrategia"] == "aleatorio"].groupby("f")[["S", "f_efectiva", "E_rel"]].mean()
    ad = curvas[e][curvas[e]["estrategia"] == "eb_recalculada"]
    fc[e] = {
        "f_c adaptativo": pr.f_critica(ad["f_efectiva"], ad["S"], u),
        "f_c azar": pr.f_critica(azar["f_efectiva"], azar["S"], u),
        "E con 5% retirado, adaptativo": ad.loc[ad["f"].round(6) == 0.05, "E_rel"].iloc[0] - 1,
        "E con 5% retirado, azar": azar.loc[azar.index.round(6) == 0.05, "E_rel"].iloc[0] - 1,
    }
pd.concat(curvas.values()).to_csv(RUTAS["salida"] / "curvas_resiliencia.csv", index=False)
pd.DataFrame(fc).to_csv(RUTAS["salida"] / "fc.csv")
pd.DataFrame(fc).map(lambda v: f"{v:.1%}")

In [ ]:
pd.concat({e: c.pivot_table(index="f", columns="estrategia", values="S") for e, c in curvas.items()},
          axis=1).iloc[::5].round(3)

## 7. Robustez R (3 réplicas, mismo método que Gvial)

In [ ]:
R = {}
for e, G in REDES.items():
    k = G.number_of_nodes()
    R[e] = [pr.robustez(G, offset=o, k=k, **PARAM["robustez"])[0] for o in PARAM["offsets_replicas"]]
tabla_R = pd.DataFrame(R, index=[f"offset {o}" for o in PARAM["offsets_replicas"]])
tabla_R.to_csv(RUTAS["salida"] / "robustez_R.csv")
tabla_R.round(4)

## 8. Comparación con Gvial

Métricas relativas, independientes del tamaño de la red. Gvial se lee de las salidas de `01_gvial`.

In [ ]:
fc_gvial = pd.read_csv(RUTAS["gvial"] / "curvas_resiliencia.csv")   # pasos de 1%, igual que aquí
azar_v = fc_gvial[fc_gvial["estrategia"] == "aleatorio"].groupby("f")[["S", "f_efectiva"]].mean()
ad_v = fc_gvial[fc_gvial["estrategia"] == "eb_recalculada"]
tramos_v = pd.read_csv(RUTAS["gvial"] / "tramos_arterial.csv")

resumen = pd.DataFrame({
    "Gvial arterial": {
        "nodos": tramos_v[["u", "v"]].stack().nunique(), "aristas": len(tramos_v),
        "Gini intermediación": pr.concentracion(tramos_v["eb"])["gini"],
        "1% superior": pr.concentracion(tramos_v["eb"])["top1"],
        "f_c adaptativo": pr.f_critica(ad_v["f_efectiva"], ad_v["S"], u),
        "f_c azar": pr.f_critica(azar_v["f_efectiva"], azar_v["S"], u),
    },
    **{f"Gtransporte {e}": {
        "nodos": REDES[e].number_of_nodes(), "aristas": REDES[e].number_of_edges(),
        "Gini intermediación": conc[e]["gini"], "1% superior": conc[e]["top1"],
        "f_c adaptativo": fc[e]["f_c adaptativo"], "f_c azar": fc[e]["f_c azar"],
    } for e in REDES},
})
resumen.loc["azar / adaptativo"] = resumen.loc["f_c azar"] / resumen.loc["f_c adaptativo"]
resumen.to_csv(RUTAS["salida"] / "comparacion_redes.csv")
resumen.round(3)

## 9. Intervención: Metro Línea 1

La Línea 1 se agrega al espacio L con sus 16 estaciones, tomadas de OpenStreetMap (red "Metro de Bogotá", extracción del 2026-11-09 en `data/raw/metro_l1/`). Las estaciones se ordenan desde Gibraltar, la terminal occidental. Cada tramo mide la distancia recta entre estaciones multiplicada por la razón entre la longitud de la vía y la suma de esas distancias. El tiempo se reparte en proporción a la longitud, de modo que el recorrido completo dure 27 minutos. Cada estación de metro se une con las de TransMilenio a 400 m o menos mediante un transbordo de 5 minutos.

La eficiencia se mide solo entre estaciones de TransMilenio, para comparar el mismo conjunto de viajes antes y después. Como la intermediación es exacta, R es determinista: las 10 réplicas dan el mismo valor.

In [ ]:
import metro_l1 as ml

METRO = {
    "geojson": RUTAS["raw"].parent / "metro_l1" / "osm_metro_l1_2026-11-09.geojson",
    "inicio": "Gibraltar",
    "minutos_recorrido": 27,
    "radio_transbordo_m": 400,
    "segundos_transbordo": 300,
    "sensibilidad_transbordo_s": [180, 300, 600],
    "offsets": list(range(0, 1000, 100)),
}
est_metro = ml.estaciones_osm(METRO["geojson"], METRO["inicio"])
factor = ml.factor_trazado(METRO["geojson"], est_metro)
GM, transbordos = ml.agregar_metro(GL, est_metro, METRO["minutos_recorrido"], factor,
                                   METRO["radio_transbordo_m"], METRO["segundos_transbordo"])
pr.validar_contrato(GM)
print(f"Factor vía / recta: {factor:.3f} | longitud de vía: {est_metro['dist_sig_m'].sum() * factor / 1000:.2f} km")
print(" > ".join(est_metro["nombre"]))
transbordos[["metro", "transmilenio", "distancia_m"]].round(0)

In [ ]:
tm = set(GL.nodes)
k_L, k_M = GL.number_of_nodes(), GM.number_of_nodes()

def evaluar(G, k):
    """Eficiencia entre estaciones de TransMilenio, R con 10 réplicas y f_c adaptativo."""
    reps = [pr.robustez(G, offset=o, k=k, **PARAM["robustez"])[0] for o in METRO["offsets"]]
    E = pr.eficiencia(G, list(tm), destinos=tm)
    curva, _ = pr.ataque(G, "eb_recalculada", list(tm), E, A["pasos"], A["frac"], k_recalculo=k)
    return {"eficiencia": E, "R_reps": reps,
            "f_c adaptativo": pr.f_critica(curva["f_efectiva"], curva["S"], PARAM["umbral_fc"])}

ev = {"TransMilenio": evaluar(GL, k_L), "TransMilenio + Metro L1": evaluar(GM, k_M)}
comp_metro = pr.comparar_replicas(ev["TransMilenio"]["R_reps"], ev["TransMilenio + Metro L1"]["R_reps"])

eb_M = pr.intermediacion(GM, "travel_time")
carga_M = pr.corredores(pr.tabla_tramos(GM, {"eb": eb_M}), {"carga": "eb"}, min_km=PARAM["min_km_corredor"])["carga"]
carga_L = corr_L["carga_t"]

tabla_metro = pd.DataFrame({
    nombre: {"eficiencia entre estaciones TM": v["eficiencia"], "R (media de 10)": pd.Series(v["R_reps"]).mean(),
             "desviación de R": pd.Series(v["R_reps"]).std(), "f_c adaptativo": v["f_c adaptativo"],
             "carga Troncal 7": (carga_M if "Metro" in nombre else carga_L).get("Troncal 7")}
    for nombre, v in ev.items()})
tabla_metro["cambio_%"] = 100 * (tabla_metro.iloc[:, 1] / tabla_metro.iloc[:, 0] - 1)
tabla_metro.to_csv(RUTAS["salida"] / "metro_l1.csv")
print("Cambio de R pareado:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in comp_metro.items()})
tabla_metro.round(4)

### Sensibilidad al tiempo de transbordo

In [ ]:
filas = []
for s in METRO["sensibilidad_transbordo_s"]:
    G_s, _ = ml.agregar_metro(GL, est_metro, METRO["minutos_recorrido"], factor, METRO["radio_transbordo_m"], s)
    filas.append({"transbordo_min": s / 60,
                  "eficiencia_%": 100 * (pr.eficiencia(G_s, list(tm), destinos=tm) / ev["TransMilenio"]["eficiencia"] - 1),
                  "R_%": 100 * (pr.robustez(G_s, k=k_M, **PARAM["robustez"])[0] / pd.Series(ev["TransMilenio"]["R_reps"]).mean() - 1)})
sens = pd.DataFrame(filas)
sens.to_csv(RUTAS["salida"] / "metro_l1_sensibilidad.csv", index=False)
sens.round(2)